In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
home_path = '/gws/ssde/j25a/duicv/yuansun/'

In [11]:
period_list = ['summer', 'winter'] 
factor_list = ['base', 'sub0.8', 'sub0.4', 'sub0.2', 'sub0.1', 'add0.1', 'add0.2', 'add0.4', 'add0.8']
factor_label_list = ['', '-80%', '-40%', '-20%', '-10%', '+10%', '+20%', '+40%', '+80%']
factor_list2= [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
factor_label_list2= ['-10%', '-20%', '-30%', '-40%', '-50%', '-60%']

In [3]:
date_list = ['2022-07-16-03600', '2022-12-10-03600']
aadt_result_list = []
GRIDNAME='UK-MCR'
var4 = ['TSA_U', 'RH2M', 'TRAFFICFLUX']
season_list = ['summer', 'winter']
start_date_list2 = ['2022-07-16T01:00:00', '2022-12-10T01:00:00']
end_date_list = ['2022-07-23-T00:00:00', '2022-12-17T00:00:00']

In [12]:
for p, period in enumerate(period_list):
    season = season_list[p]
    casename = f'{GRIDNAME}_{season}'
    date = date_list[p]
    for factor in factor_list + factor_list2 :
        if factor == 'base':
            ds_traffic = xr.open_dataset(f'{home_path}0_urban_traffic/archive/{GRIDNAME}_traffic/lnd/hist/{GRIDNAME}_traffic.clm2.h0.2022-01-01-03600.nc')
            ds_traffic = ds_traffic.assign_coords(time=ds_traffic.time.dt.round("h"))
            ds_factor = ds_traffic.sel(time=slice(start_date_list2[p], end_date_list[p]))
        elif factor in factor_list2:
            ds_factor = xr.open_dataset(f'{home_path}0_urban_traffic/archive/sensitivity2/0_land_output/{casename}2/{casename}.clm2.h0.{date}_{factor}.nc')
            ds_factor = ds_factor.assign_coords(time=ds_factor.time.dt.round("h"))
        else:    
            ds_factor = xr.open_dataset(f'{home_path}0_urban_traffic/archive/sensitivity/0_land_output/{casename}/{casename}.clm2.h0.{date}_{factor}.nc')
            ds_factor = ds_factor.assign_coords(time=ds_factor.time.dt.round("h"))
        ds_var = ds_factor[var4].to_dataframe()
        var_mean = ds_var.mean().to_dict()
        var_mean['factor'] = factor
        aadt_result_list.append((period, var_mean))
df_aadt_result = pd.DataFrame([{'period': x[0], **x[1]} for x in aadt_result_list])
df_aadt_result['TSA_U'] -=273.15
df_aadt_result['factor_label'] = df_aadt_result['factor'].map({**dict(zip(factor_list, factor_label_list)), **dict(zip(factor_list2, factor_label_list2))})
df_aadt_result.to_csv('data_for_figure/UK-Manchester_results.csv', index=False)
df_aadt_result

,period,TSA_U,RH2M,TRAFFICFLUX,factor,factor_label
0,summer,21.437250,74.060966,16.253613,base,
1,summer,21.327692,74.755310,3.250720,sub0.8,-80%
2,summer,21.386346,74.399185,9.752160,sub0.4,-40%
3,summer,21.413965,74.232178,13.002880,sub0.2,-20%
4,summer,21.426630,74.135902,14.628241,sub0.1,-10%
...,...,...,...,...,...,...
145,winter,-0.200598,85.149681,13.677839,0.1,-20%
146,winter,-0.216895,85.267159,12.195906,0.15,-30%
147,winter,-0.233466,85.405495,10.713970,0.2,-40%
148,winter,-0.252203,85.501892,9.232037,0.25,-50%
